# Basic Function Calling with AI Agent

This notebook demonstrates how to implement basic function calling with an AI agent that responds to name-related queries.

## Overview
- Create a `fetch_name` function
- Define function schema for the AI
- Implement an agent workflow
- Handle name-related queries

## 1. Setup and Dependencies

In [ ]:
# Install required packages (uncomment if needed)
# !pip install anthropic openai python-dotenv

In [ ]:
import json
import os
from typing import Dict, List, Any, Optional

# For API keys (optional - you can also set them directly)
# from dotenv import load_dotenv
# load_dotenv()

## 2. Define the fetch_name Function

This function simulates a database lookup for person information based on their ID.

In [ ]:
# Simulated database of people
PEOPLE_DATABASE = {
    "user_001": {
        "name": "Alice Johnson",
        "role": "Software Engineer",
        "department": "Engineering"
    },
    "user_002": {
        "name": "Bob Smith",
        "role": "Product Manager",
        "department": "Product"
    },
    "user_003": {
        "name": "Carol Williams",
        "role": "Data Scientist",
        "department": "Analytics"
    },
    "user_004": {
        "name": "David Brown",
        "role": "UX Designer",
        "department": "Design"
    }
}

def fetch_name(user_id: str) -> Dict[str, Any]:
    """
    Fetch user information by user ID.
    
    Args:
        user_id: The unique identifier for the user
        
    Returns:
        Dictionary containing user information including name, role, and department
    """
    if user_id in PEOPLE_DATABASE:
        return {
            "success": True,
            "data": PEOPLE_DATABASE[user_id]
        }
    else:
        return {
            "success": False,
            "error": f"User ID '{user_id}' not found in database"
        }

# Test the function
print("Testing fetch_name function:")
print(json.dumps(fetch_name("user_001"), indent=2))
print(json.dumps(fetch_name("user_999"), indent=2))

## 3. Define Function Schema for AI

We need to describe our function in a format that the AI can understand.

In [ ]:
# Function schema for Anthropic Claude
TOOLS = [
    {
        "name": "fetch_name",
        "description": "Fetches user information including name, role, and department based on their user ID. Use this when you need to look up information about a specific user.",
        "input_schema": {
            "type": "object",
            "properties": {
                "user_id": {
                    "type": "string",
                    "description": "The unique identifier for the user (e.g., 'user_001', 'user_002')"
                }
            },
            "required": ["user_id"]
        }
    }
]

print("Function schema defined:")
print(json.dumps(TOOLS, indent=2))

## 4. Function Executor

This utility executes the actual Python function based on the AI's tool call.

In [ ]:
def execute_function(function_name: str, function_args: Dict[str, Any]) -> Any:
    """
    Execute a function by name with given arguments.
    
    Args:
        function_name: Name of the function to execute
        function_args: Dictionary of arguments to pass to the function
        
    Returns:
        Result of the function execution
    """
    available_functions = {
        "fetch_name": fetch_name
    }
    
    if function_name in available_functions:
        return available_functions[function_name](**function_args)
    else:
        return {"error": f"Function '{function_name}' not found"}

# Test the executor
print("Testing function executor:")
result = execute_function("fetch_name", {"user_id": "user_002"})
print(json.dumps(result, indent=2))

## 5. Agent Workflow with Anthropic Claude

Implement the agent that can use function calling to respond to queries.

In [ ]:
try:
    from anthropic import Anthropic
    
    class FunctionCallingAgent:
        """
        An AI agent that can use function calling to respond to user queries.
        """
        
        def __init__(self, api_key: Optional[str] = None):
            """
            Initialize the agent with Anthropic API.
            
            Args:
                api_key: Anthropic API key (if None, uses ANTHROPIC_API_KEY env var)
            """
            self.client = Anthropic(api_key=api_key)
            self.model = "claude-3-5-sonnet-20241022"
            
        def run(self, user_query: str, max_iterations: int = 5) -> str:
            """
            Process a user query using function calling.
            
            Args:
                user_query: The user's question or request
                max_iterations: Maximum number of turns to prevent infinite loops
                
            Returns:
                The final response to the user
            """
            messages = [
                {"role": "user", "content": user_query}
            ]
            
            print(f"\n{'='*60}")
            print(f"User Query: {user_query}")
            print(f"{'='*60}\n")
            
            for iteration in range(max_iterations):
                print(f"Iteration {iteration + 1}:")
                
                # Call Claude with tools
                response = self.client.messages.create(
                    model=self.model,
                    max_tokens=1024,
                    tools=TOOLS,
                    messages=messages
                )
                
                print(f"  Stop reason: {response.stop_reason}")
                
                # Check if we're done
                if response.stop_reason == "end_turn":
                    # Extract text response
                    final_response = ""
                    for block in response.content:
                        if hasattr(block, "text"):
                            final_response += block.text
                    print(f"\n{'='*60}")
                    print(f"Final Response: {final_response}")
                    print(f"{'='*60}\n")
                    return final_response
                
                # Handle tool use
                if response.stop_reason == "tool_use":
                    # Add assistant's response to messages
                    messages.append({
                        "role": "assistant",
                        "content": response.content
                    })
                    
                    # Process tool calls
                    tool_results = []
                    for block in response.content:
                        if block.type == "tool_use":
                            print(f"  Tool called: {block.name}")
                            print(f"  Arguments: {block.input}")
                            
                            # Execute the function
                            result = execute_function(block.name, block.input)
                            print(f"  Result: {result}")
                            
                            tool_results.append({
                                "type": "tool_result",
                                "tool_use_id": block.id,
                                "content": json.dumps(result)
                            })
                    
                    # Add tool results to messages
                    messages.append({
                        "role": "user",
                        "content": tool_results
                    })
                    
                print()
            
            return "Max iterations reached without completion."
    
    print("✓ Anthropic agent class defined successfully")
    
except ImportError:
    print("⚠ Anthropic library not installed. Run: pip install anthropic")
    print("⚠ Skipping Anthropic agent implementation")

## 6. Alternative: Agent Workflow with OpenAI

For comparison, here's the same functionality using OpenAI's API.

In [ ]:
try:
    from openai import OpenAI
    
    class OpenAIFunctionCallingAgent:
        """
        An AI agent using OpenAI's function calling.
        """
        
        def __init__(self, api_key: Optional[str] = None):
            """
            Initialize the agent with OpenAI API.
            
            Args:
                api_key: OpenAI API key (if None, uses OPENAI_API_KEY env var)
            """
            self.client = OpenAI(api_key=api_key)
            self.model = "gpt-4-turbo-preview"
            
        def run(self, user_query: str, max_iterations: int = 5) -> str:
            """
            Process a user query using function calling.
            """
            messages = [
                {"role": "user", "content": user_query}
            ]
            
            # Convert tools to OpenAI format
            openai_tools = [
                {
                    "type": "function",
                    "function": {
                        "name": tool["name"],
                        "description": tool["description"],
                        "parameters": tool["input_schema"]
                    }
                }
                for tool in TOOLS
            ]
            
            print(f"\n{'='*60}")
            print(f"User Query: {user_query}")
            print(f"{'='*60}\n")
            
            for iteration in range(max_iterations):
                print(f"Iteration {iteration + 1}:")
                
                response = self.client.chat.completions.create(
                    model=self.model,
                    messages=messages,
                    tools=openai_tools
                )
                
                message = response.choices[0].message
                
                # Check if we're done
                if not message.tool_calls:
                    print(f"\n{'='*60}")
                    print(f"Final Response: {message.content}")
                    print(f"{'='*60}\n")
                    return message.content
                
                # Add assistant's message
                messages.append(message)
                
                # Process tool calls
                for tool_call in message.tool_calls:
                    function_name = tool_call.function.name
                    function_args = json.loads(tool_call.function.arguments)
                    
                    print(f"  Tool called: {function_name}")
                    print(f"  Arguments: {function_args}")
                    
                    result = execute_function(function_name, function_args)
                    print(f"  Result: {result}")
                    
                    messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "content": json.dumps(result)
                    })
                
                print()
            
            return "Max iterations reached without completion."
    
    print("✓ OpenAI agent class defined successfully")
    
except ImportError:
    print("⚠ OpenAI library not installed. Run: pip install openai")
    print("⚠ Skipping OpenAI agent implementation")

## 7. Test the Agent

Now let's test our function calling agent with various queries.

In [ ]:
# Set your API key here or use environment variable
# For Anthropic:
# os.environ["ANTHROPIC_API_KEY"] = "your-api-key-here"

# For OpenAI:
# os.environ["OPENAI_API_KEY"] = "your-api-key-here"

print("API Key Setup:")
print(f"ANTHROPIC_API_KEY: {'Set' if os.getenv('ANTHROPIC_API_KEY') else 'Not set'}")
print(f"OPENAI_API_KEY: {'Set' if os.getenv('OPENAI_API_KEY') else 'Not set'}")

In [ ]:
# Test with Anthropic Claude
try:
    agent = FunctionCallingAgent()
    
    # Example 1: Simple name lookup
    response = agent.run("What is the name of user_001?")
    
except NameError:
    print("Anthropic agent not available. Install with: pip install anthropic")
except Exception as e:
    print(f"Error: {e}")
    print("Make sure to set your ANTHROPIC_API_KEY environment variable")

In [ ]:
# More test queries
try:
    agent = FunctionCallingAgent()
    
    test_queries = [
        "Tell me about user_002",
        "What role does user_003 have?",
        "Which department is user_004 in?",
        "Can you find information about user_999?"  # This should handle the error gracefully
    ]
    
    for query in test_queries:
        print("\n" + "#" * 60)
        response = agent.run(query)
        
except NameError:
    print("Agent not available.")
except Exception as e:
    print(f"Error: {e}")

## 8. Standalone Example (No API Required)

Here's a simplified mock version that demonstrates the concept without requiring API keys.

In [ ]:
class MockAgent:
    """
    A mock agent that simulates function calling behavior for demonstration.
    This doesn't require any API keys.
    """
    
    def simulate_query(self, user_query: str) -> str:
        """
        Simulate processing a user query with function calling.
        """
        print(f"\n{'='*60}")
        print(f"User Query: {user_query}")
        print(f"{'='*60}\n")
        
        # Simple pattern matching to determine if we need to call fetch_name
        user_id = None
        for possible_id in ["user_001", "user_002", "user_003", "user_004"]:
            if possible_id in user_query.lower():
                user_id = possible_id
                break
        
        if user_id:
            print("Step 1: Detected need to fetch user information")
            print(f"Step 2: Calling fetch_name(user_id='{user_id}')")
            
            result = fetch_name(user_id)
            print(f"Step 3: Received result: {json.dumps(result, indent=2)}")
            
            if result["success"]:
                data = result["data"]
                response = f"The user {user_id} is {data['name']}, who works as a {data['role']} in the {data['department']} department."
            else:
                response = f"Sorry, I couldn't find information for {user_id}."
            
            print(f"\nStep 4: Generated response")
            print(f"{'='*60}")
            print(f"Final Response: {response}")
            print(f"{'='*60}\n")
            
            return response
        else:
            response = "I couldn't identify a specific user ID in your query. Please include a user ID like 'user_001'."
            print(f"Final Response: {response}\n")
            return response

# Test the mock agent
mock_agent = MockAgent()

test_queries = [
    "What is the name of user_001?",
    "Tell me about user_003",
    "What role does user_002 have?"
]

for query in test_queries:
    mock_agent.simulate_query(query)

## Summary

This notebook demonstrates:

1. **Function Definition**: Created `fetch_name()` function that retrieves user data
2. **Function Schema**: Defined the function in a format AI models can understand
3. **Function Executor**: Built a dispatcher to execute functions based on AI requests
4. **Agent Workflow**: Implemented complete agents using both Anthropic and OpenAI APIs
5. **Mock Example**: Provided a standalone example that works without API keys

### Key Concepts:

- **Function Calling** allows AI models to use external functions to gather information
- The AI decides when to call functions based on user queries
- Results are fed back to the AI to generate natural language responses
- This creates a powerful agentic workflow that can interact with external systems

### Next Steps:

1. Add more functions (e.g., `search_by_name`, `update_user`, `list_all_users`)
2. Connect to a real database instead of the mock data
3. Add error handling and validation
4. Implement logging and monitoring
5. Create a web interface or CLI for the agent